In [1]:
from hyrax import Hyrax
import numpy as np

h = Hyrax()

data_request = {
    "visualize": {
        "results": {
            "dataset_class": "ResultDataset",
            "data_location": "/Users/drew/code/hyrax/docs/pre_executed/results/synthetic-10M_b",
            "primary_id_field": "object_id",
        },
    }
}

h.set_config("data_request", data_request)

ds = h.prepare()
vis_ds = ds["visualize"].prepped_datasets["results"]

[2026-04-17 10:52:11,067 hyrax.verbs.prepare:INFO] Finished Prepare


In [ ]:
%%timeit
a = vis_ds.__getitem__(np.array(range(0, 10)))

In [ ]:
%%timeit
a = vis_ds.__getitem__(np.array(range(0, 100)))

In [ ]:
%%timeit
a = vis_ds.__getitem__(np.array(range(0, 1000)))

In [ ]:
%%timeit
a = vis_ds.__getitem__(np.array(range(0, 100_000)))

In [ ]:
%%timeit
a = vis_ds.__getitem__(np.array(range(0, 1_000_000)))

In [ ]:
idx = np.array(range(0, 50_000_000))
table_len = 10_000_000

In [ ]:
from pathlib import Path
import lancedb

LANCE_DB_DIR = "lance_db"
TABLE_NAME = "results"


data_location = "/Users/drew/code/hyrax/docs/pre_executed/results/synthetic-100M"

data_location = Path(data_location)
lance_dir = data_location / LANCE_DB_DIR

db = lancedb.connect(str(lance_dir))
table = db.open_table(TABLE_NAME)

# Get the underlying lance dataset for efficient access
lance_dataset = table.to_lance()

In [ ]:
%%timeit
result = lance_dataset.take(idx)

In [2]:
import json
from pathlib import Path
import lancedb

LANCE_DB_DIR = "lance_db"
TABLE_NAME = "results"


data_location = "/Users/drew/code/hyrax/docs/pre_executed/results/synthetic-100M_b"

data_location = Path(data_location)
lance_dir = data_location / LANCE_DB_DIR

db = lancedb.connect(str(lance_dir))
table = db.open_table(TABLE_NAME)

# Get the underlying lance dataset for efficient access
lance_dataset = table.to_lance()
schema_metadata = table.schema.metadata
tensor_shape = json.loads(schema_metadata[b"tensor_shape"].decode("utf-8"))

tensor_dtype = np.dtype(schema_metadata[b"tensor_dtype"].decode("utf-8"))

In [ ]:
%%timeit
result = lance_dataset.take(idx)

In [ ]:
data_column = lance_dataset.to_table(columns=["data"])

In [ ]:
tensors = np.array(data_column.to_pylist(), dtype=tensor_dtype).reshape([-1] + tensor_shape)

In [6]:
arrow_col = lance_dataset.to_table(columns=["data"])["data"]

# If ChunkedArray (multiple fragments), combine first
if hasattr(arrow_col, "combine_chunks"):
    arrow_col = arrow_col.combine_chunks()

# Access the flat buffer directly — no Python objects created
flat_np = arrow_col.values.to_numpy(zero_copy_only=False)

# Reshape is O(1) (a view, no data copy)
tensors = flat_np.reshape(-1, *tensor_shape).astype(tensor_dtype, copy=False)

In [7]:
tensors[0]

array([-1.940473,  8.256199], dtype=float32)

In [8]:
len(tensors)

100000000